# 02 · Experimentos: zero-shot × few-shot × chain-of-thought

Compara **estratégias de prompt** e **modelos** na mesma tarefa de extração estruturada. O notebook executa automaticamente os provedores cujas credenciais estão no ambiente (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`); sem chaves, roda só o baseline. Resultados ficam em cache em `results/runs.jsonl`, então reexecutar é barato.

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


In [2]:
from promptlab.prompts import build, STRATEGIES
sample = 'Numero do documento: 000123\nPeso bruto: 1.200,5 kg'
for s in STRATEGIES:
    system, user = build(s, sample)
    print(f'##### {s}  ({len(user)} caracteres de prompt)')
    print(user[-700:] if s != 'few_shot' else user[:900] + '\n[...]')
    print()

##### zero_shot  (1458 caracteres de prompt)
decimal, sem símbolo)
- "valor_frete": valor do frete em reais (número com ponto decimal, sem símbolo)
- "placa_veiculo": placa do veículo no formato AAA0A00 ou AAA-0000; null se não informada

Regras:
1. Use exatamente as chaves listadas, na mesma ordem.
2. Números como números JSON (ponto decimal, sem 'R$', sem separador de milhar).
3. Datas em ISO (AAAA-MM-DD). CNPJ no formato 00.000.000/0000-00.
4. Corrija confusões óbvias de OCR (letra O no lugar do dígito 0).
5. Se o peso estiver em toneladas, converta para kg.
6. Campo ausente no documento => null (não deduza).

=== DOCUMENTO ===
Numero do documento: 000123
Peso bruto: 1.200,5 kg
=== FIM ===

Responda apenas com um objeto JSON válido.

##### few_shot  (5124 caracteres de prompt)
Veja exemplos resolvidos e depois extraia o documento final.

### Exemplo 1
=== DOCUMENTO ===
DACTE  -  CONHECIMENTO  DE  TRANSPORTE
N DOC.:  67498O   EMISSAO:  28/11/2026
REMET.:  SERTÃO EQUIPAMENTOS MÉDICOS 

## Configuração da bateria
Defina os modelos por variável de ambiente, p.ex.:
`LAB_MODELS="anthropic:<modelo> openai:gpt-4o-mini"`. Tamanho da amostra em `LAB_N`.

In [3]:
from promptlab.dataset import load_jsonl, generate, save_jsonl
from promptlab.runner import run_experiments
data_path = ROOT / 'data' / 'eval_docs.jsonl'
if not data_path.exists():
    save_jsonl(generate(150, seed=42), data_path)
N = int(os.getenv('LAB_N', '150'))
docs = load_jsonl(data_path)[:N]
models = ['baseline'] + os.getenv('LAB_MODELS', '').split()
print('modelos:', models, '| documentos:', len(docs))

modelos: ['baseline'] | documentos: 150


In [4]:
rows = run_experiments(docs, models, STRATEGIES, ROOT / 'results' / 'runs.jsonl', workers=4)
print(f'{len(rows)} execuções registradas')

150 execuções registradas


In [5]:
from promptlab.evaluate import results_to_frame, summarize
keep = {d.doc_id for d in docs}
df = results_to_frame([r for r in rows if r['doc_id'] in keep])
summarize(df).round(3)

,model,strategy,docs,exact_match,json_valid,hallucination_rate,latency_p50_s,cost_per_1k_docs_usd,field_accuracy
0,baseline:regex,regex,150,0.327,1.0,0.0,0.0,0.0,0.343
